In [1]:
import sys
sys.path.append('../')

from utils_ferm import (
    orthogonal_transform_obt_tbt,
    obt_phys_spatial_to_spin,
    tbt_phys_spatial_to_spin,
    make_short_H_ferm_op
)
from utils_states import (
    convert_TZ_format_to_sparse_format,
    convert_dense_format_to_sparse_format,
    tz_state_seniority_config,
    compress_state,
    decompress_state
)
from utils_m1_seniority import (
    project_out_seniority_symmetries
)
from utils_m2_factorize import (
    expand_tensor_product,
    expand_tensor_product_for_incomplete_qubit_set,
    get_indices_mapping_2_wvn,
    factorize_state,
    evaluate_fully_classical_factors
)
from openfermion import (
    get_sparse_operator,
    jordan_wigner
)

import numpy as np
import pickle

In [2]:
# load Q-SENSE basis states

molecule    = 'h2o'
bond_length = 1.0
filename    = f'../{molecule}_data/Uext_CSF_for_Praveen_Smik_{bond_length}.dump'

with open(filename, 'rb') as f:
    (
    list_list_refCSF,
    list_list_Uext_mp2_CSF,
    list_list_Uext_mp2_ampld,
    list_list_Uext_opt_ampld,
    list_orb_rot,
    x_orbrot,
    Enuc,
    obt_spatial,
    tbt_spatial
    ) = pickle.load(f)

# rotate orbitals and obtain Hamiltonian operator

if len(list_orb_rot) != 0:
    obt, tbt = orthogonal_transform_obt_tbt(x_orbrot,list_orb_rot,obt_spatial,tbt_spatial)
else:
    obt = obt_phys_spatial_to_spin(obt_spatial)
    tbt = tbt_phys_spatial_to_spin(tbt_spatial)

Hfer    = make_short_H_ferm_op(Enuc, obt, tbt)
Hqub    = jordan_wigner(Hfer)
Hsparse = get_sparse_operator(Hqub)

Nqubits = obt.shape[0]
Norb    = Nqubits // 2
dim     = 2 ** Nqubits

# obtain relevant information about Q-SENSE states (UCSFs, CSFs, W information) in a linear list

UCSF_tz_states = []
CSF_tz_states  = []
W_amplitudes   = []

for i, ucsf_list in enumerate(list_list_Uext_mp2_CSF):
    for j, ucsf in enumerate(ucsf_list):
        UCSF_tz_states.append(ucsf)
        CSF_tz_states.append(list_list_refCSF[i][j])
        W_amplitudes.append(list_list_Uext_mp2_ampld[i])

# process information so that we can taper and factorize the Q-SENSE states

Nstates          = len(UCSF_tz_states)
configs          = [tz_state_seniority_config(tz_state) for tz_state in UCSF_tz_states]
UCSF_information = [get_indices_mapping_2_wvn(CSF_tz_states[i], W_amplitudes[i], Norb) for i in range(Nstates)]

SW_list          = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'W']) for i in range(Nstates)]
SV_list          = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'V']) for i in range(Nstates)]
SN_list          = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'N']) for i in range(Nstates)]
state_type_list  = [UCSF_information[i][1] for i in range(Nstates)]

# taper and factorize the Q-SENSE basis states

statevectors                    = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in UCSF_tz_states]
tapered_statevectors            = [convert_dense_format_to_sparse_format(compress_state(psi.toarray()[0])) for psi in statevectors]
factorized_tapered_statevectors = [factorize_state(tapered_statevectors[i], SW_list[i], SV_list[i], SN_list[i], state_type_list[i]) 
                                   for i in range(Nstates)]

# verify that all three states are the same by converting between them

statevectors_recov = [decompress_state(tapered_statevectors[i].toarray()[0], configs[i]) for i in range(Nstates)]
for i in range(Nstates):
    assert np.allclose(statevectors[i].toarray()[0], statevectors_recov[i])

tapered_statevectors_recov = [expand_tensor_product_for_incomplete_qubit_set(psi_t_FD) for psi_t_FD in factorized_tapered_statevectors]
for i in range(Nstates):
    assert np.allclose(tapered_statevectors[i].toarray()[0], tapered_statevectors_recov[i])

In [3]:
# obtain subspace Hamiltonian using three kinds of states

Hsub_full = np.zeros([Nstates, Nstates], dtype=np.complex128)

for i in range(Nstates):
    state          = statevectors[i]
    Hsub_full[i,i] = (state @ Hsparse @ state.T)[0,0]

for i in range(Nstates):
    for j in range(Nstates):
        if i > j:
            bra = statevectors[i]
            ket = statevectors[j]
            Hsub_full[i,j] = (bra @ Hsparse @ ket.T)[0,0]
            Hsub_full[j,i] = (bra @ Hsparse @ ket.T)[0,0]

Hsub_tapered = np.zeros([Nstates, Nstates], dtype=np.complex128)

for i in range(Nstates):
    print(f'Tapered: {i, i}', end='\r')
    state             = tapered_statevectors[i]
    config            = configs[i]
    Htapered          = get_sparse_operator(project_out_seniority_symmetries(Hqub, Nqubits, config, config), Norb)
    Hsub_tapered[i,i] = (state @ Htapered @ state.T)[0,0]

for i in range(Nstates):
    for j in range(Nstates):
        if i > j:
            print(f'Tapered : {i, j}', end='\r')
            bra               = tapered_statevectors[i]
            bra_config        = configs[i]
            ket               = tapered_statevectors[j]
            ket_config        = configs[j]

            Htapered          = get_sparse_operator(project_out_seniority_symmetries(Hqub, Nqubits, bra_config, ket_config), Norb)
            Hsub_tapered[i,j] = (bra @ Htapered @ ket.T)[0,0]
            Hsub_tapered[j,i] = (bra @ Htapered @ ket.T)[0,0]

Hsub_factorized = np.zeros([Nstates, Nstates], dtype=np.complex128)

for i in range(Nstates):
    print(f'Factorized : {i, i}', end='\r')
    state                = factorized_tapered_statevectors[i]
    state_labels         = UCSF_information[i][0]
    state_config         = configs[i]

    Htapered             = project_out_seniority_symmetries(Hqub, Nqubits, state_config, state_config)

    HtQ, statetQ, _, NtQ = evaluate_fully_classical_factors(state, state, state_labels, state_labels, Htapered)

    if NtQ == 0:
        Hsub_factorized[i,i] = HtQ.constant

    else:
        statetQ              = convert_dense_format_to_sparse_format(statetQ)
        Hsub_factorized[i,i] = (statetQ @ get_sparse_operator(HtQ, NtQ) @ statetQ.T)[0,0]

for i in range(Nstates):
    for j in range(Nstates):
        if i > j:
            print(f'Factorized : {i, j}', end='\r')
            bra                    = factorized_tapered_statevectors[i]
            bra_labels             = UCSF_information[i][0]
            bra_config             = configs[i]

            ket                    = factorized_tapered_statevectors[j]
            ket_labels             = UCSF_information[j][0]
            ket_config             = configs[j]

            Htapered               = project_out_seniority_symmetries(Hqub, Nqubits, bra_config, ket_config)
            HtQ, bratQ, kettQ, NtQ = evaluate_fully_classical_factors(bra, ket, bra_labels, ket_labels, Htapered)

            if NtQ == 0:
                Hsub_factorized[i,j] = HtQ.constant
                Hsub_factorized[j,i] = HtQ.constant

            else:
                bratQ                = convert_dense_format_to_sparse_format(bratQ)
                kettQ                = convert_dense_format_to_sparse_format(kettQ)
                HtQ                  = get_sparse_operator(HtQ, NtQ)
                Hsub_factorized[i,j] = (bratQ @ HtQ @ kettQ.T)[0,0]
                Hsub_factorized[j,i] = (bratQ @ HtQ @ kettQ.T)[0,0]



In [7]:
assert np.allclose(Hsub_full, Hsub_tapered)
assert np.allclose(Hsub_full, Hsub_factorized)
assert np.allclose(Hsub_tapered, Hsub_factorized)